# Tutorial 3: Model-Observation comparison with ROMS and CrocoLake

The goal of this tutorial is to learn how to generate your own [obs_seq files](https://docs.dart.ucar.edu/en/latest/guide/detailed-structure-obs-seq.html) from CrocoLake and use them in a model-obs comparison workflow. We will select a specific region, interpolate our model output to the temperature and salinity values of CrocoLake data in that region, and compare the interpolated model and the observations.

## Generating your own obs_seq.in files

In [ ]:
crocolake_path = '$CROCOLAKE_PATH/0007_PHY_CROCOLAKE-QC-MERGED'

import os
basename = "myCL_obs_seq_"
outdir = "/home/enrico/myWHOI/CrocoLake/model2obs/roms_tests/input/test_2/in_CrocoLake/"
basename = os.path.expandvars(outdir+basename)
outdir = os.path.expandvars(outdir)
crocolake_path = os.path.expandvars(crocolake_path)

In [ ]:
import datetime
from convert_crocolake_obs import ObsSequence

# define horizontal region
LAT0 = 27
LAT1 = 52
LON0 = -82
LON1 = -35

# define variables to import from CrocoLake
selected_variables = [
    "DB_NAME",  # ARGO, GLODAP, SprayGliders, OleanderXBT, Saildrones
    "JULD", # this contains timestamp
    "LATITUDE",
    "LONGITUDE",
    "PRES", # This will be automatically converted to depths in meters
    "TEMP",
    "PRES_QC",
    "TEMP_QC",
    "PRES_ERROR",
    "TEMP_ERROR",
    "PSAL",
    "PSAL_QC",
    "PSAL_ERROR"
]

# month and year are constant in out case
year0 = 2018
month0 = 5
day0 = 31
basename = basename + f"ROMS_test2"
if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)

# we loop to generate one file per day
for j in range(5):

    # set date range
    day1 = day0+j
    day2 = day1+1

    month1 = month0
    month2 = month0

    if day1>31:
        day1 = day1-31
        month1 = month1+1
    if day2>31:
        day2 = day2-31
        month2 = month2+1
        
    date0 = datetime.datetime(year0, month1, day1, 0, 0, 0)
    date1 = datetime.datetime(year0, month2, day2, 0, 0, 0)
    print(f"Converting obs between {date0} and {date1}")

    # this defines AND filters, i.e. we want to load each observation that has latitude within the given range AND longitude within the given range, etc.
    # to exclude NaNs, impose a range to a variable
    and_filters = (
        ("LATITUDE",'>',LAT0),  ("LATITUDE",'<',LAT1),
        ("LONGITUDE",'>',LON0), ("LONGITUDE",'<',LON1),
        ("PRES",'>',-1e30), ("PRES",'<',1e30),
        ("JULD",">",date0), ("JULD","<",date1)
    )

    # this adds OR conditions to the and_filters, i.e. we want to load all observations that statisfy the AND conditions above, AND that have finite salinity OR temperature values
    db_filters = [
        list(and_filters) + [("PSAL", ">", -1e30), ("PSAL", "<", 1e30)],
        list(and_filters) + [("TEMP", ">", -1e30), ("TEMP", "<", 1e30)],
    ]

    # generate output filename
    obs_seq_out = basename + f".{year0:04d}{month1:02d}{day1:02d}.out"

    # generate obs_seq.in file
    obsSeq = ObsSequence(
        crocolake_path,
        selected_variables,
        db_filters,
        obs_seq_out=obs_seq_out,
        loose=True
    )
    obsSeq.write_obs_seq()


## Running the workflow

In [ ]:
from model2obs.workflows import WorkflowModelObs

# interpolate model onto obs space for the single float
workflow_float = WorkflowModelObs.from_config_file('config_roms_test2.yaml')
workflow_float.run(clear_output=True,trim_obs=False) #use flag clear_output=True if you want to re-run it and automatically clean all previous output


## Displaying the interactive map

We can load the data with the same tools as in Tutorial 1. Note that in this case all interpolations were succesfull, so `get_good_model_obs_df()` and `get_all_model_obs()` will return the same dataframe (`get_failed_model_obs()` will return an empty dataframe).

In [ ]:
good_model_obs_df = workflow_float.get_good_model_obs_df(compute=True) # compute=True triggers the compute of the dask dataframe, returning a pandas dataframe with data loaded in memory
good_model_obs_df.head()                                                   # displays first 5 rows in the dataframe

Now load the interactive map as in Tutorial 1. What do you see? (Again, you can try and code it yourself or load the solution and run it.)

In [ ]:
from model2obs.viz import InteractiveWidgetMap

# Create an interactive map widget to visualize model-observation comparisons
# The widget provides controls for selecting variables, observation types, and time ranges
widget = InteractiveWidgetMap(good_model_obs_df)
widget.setup()


The previous cell should show a dot in the middle of the ocean, which is not very informative. We can then use `MapConfig` to pass extra arguments to `InteractiveWidgetMap`, among which the extent of the map area to plot. Play with the `map_extent` values in the cell below and re-execute the cell until you're happy with your plotted region.

In [ ]:
from model2obs.viz import MapConfig

map_config = MapConfig(
    map_extent=(-90,0,0,90) #(lon_min, lon_max, lat_min, lat_max)
)

# Create an interactive map widget to visualize model-observation comparisons
# The widget provides controls for selecting variables, observation types, and time ranges
widget = InteractiveWidgetMap(good_model_obs_df, config=map_config)
widget.setup()

### Interactive profile

Finally, we can load the interactive profile widget to explore how our model performed compared to the specific float measurements we selected:

In [ ]:
from model2obs.viz import InteractiveWidgetProfile
# Create an interactive profile widget to analyze vertical profiles
# This is ideal for analyzing single float/CTD profiles vs model data
widget = InteractiveWidgetProfile(good_model_obs_df)
widget.setup()

You can also pass some settings through the `config` argument of the InteractiveWidgetProfile class:

In [ ]:
from model2obs.viz import ProfileConfig

# Customize the profile widget appearance and behavior
profile_config = ProfileConfig(
    figure_size=(7, 7),
    marker_size=5,
    marker_alpha=0.6,
    invert_yaxis=False,  # Don't invert for this example
    grid=True
)

# Create widget with custom configuration
widget = InteractiveWidgetProfile(good_model_obs_df, x='obs', y='interpolated_model', config=profile_config)
widget.setup()